Step 1

In [1]:
# Install required libraries
!pip install datasets tensorflow -q

In [2]:
# Import libraries
from datasets import load_dataset

# Load AG News dataset
dataset = load_dataset("wangrongsheng/ag_news")

# Display dataset information
print(dataset)



README.md:   0%|          | 0.00/8.07k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})


In [3]:
# First training sample
print(dataset["train"][0])

{'text': "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.", 'label': 2}


step 2 Clean and Normalize the Text

In [4]:
import re

# Function to clean text
def clean_text(text):
    # Convert text to lowercase
    text = text.lower()

    # Remove punctuation, numbers, and special characters
    text = re.sub(r'[^a-z\s]', '', text)

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text

# Clean the training text
train_texts = [clean_text(sample["text"]) for sample in dataset["train"]]

# Clean the testing text
test_texts = [clean_text(sample["text"]) for sample in dataset["test"]]

# Store the labels
train_labels = [sample["label"] for sample in dataset["train"]]
test_labels = [sample["label"] for sample in dataset["test"]]

# Display the first cleaned article
print(train_texts[0])

# Display the first label
print(train_labels[0])

wall st bears claw back into the black reuters reuters shortsellers wall streets dwindlingband of ultracynics are seeing green again
2


Step 3: Tokenize the Strings into Numbers

In [5]:
# Import Tokenizer for converting text into sequences of integers
from tensorflow.keras.preprocessing.text import Tokenizer

# Maximum number of words to keep in the vocabulary
vocab_size = 20000

# Create a tokenizer object
# oov_token handles words not present in the vocabulary
tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")

# Learn the vocabulary from the training text
tokenizer.fit_on_texts(train_texts)

# Convert training text into sequences of integers
X_train = tokenizer.texts_to_sequences(train_texts)

# Convert testing text into sequences of integers
X_test = tokenizer.texts_to_sequences(test_texts)

# Display the first 20 token IDs of the first training sample
print(X_train[0][:20])

[392, 325, 1526, 14261, 100, 55, 2, 813, 24, 24, 1, 392, 1989, 1, 5, 1, 35, 3894, 738, 296]


Step 4: Pad and Truncate Sequences

In [6]:
# Import pad_sequences to make all sequences the same length
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Set the maximum sequence length
max_length = 50

# Pad shorter sequences with zeros and truncate longer ones
X_train = pad_sequences(
    X_train,
    maxlen=max_length,
    padding='post',      # Add zeros at the end
    truncating='post'    # Remove extra words from the end
)

# Apply the same padding to the test data
X_test = pad_sequences(
    X_test,
    maxlen=max_length,
    padding='post',
    truncating='post'
)

# Display the shape of the padded training data
print(X_train.shape)

(120000, 50)


Step 5: Convert Labels to Categorical Vectors

In [7]:
# Import function for one-hot encoding
from tensorflow.keras.utils import to_categorical

# Number of output classes in AG News dataset
num_classes = 4

# Convert training labels into one-hot encoded vectors
y_train = to_categorical(train_labels, num_classes)

# Convert testing labels into one-hot encoded vectors
y_test = to_categorical(test_labels, num_classes)

# Display the first encoded label
print(y_train[0])

[0. 0. 1. 0.]


Step 6: Define the Simple RNN Model

In [8]:
# Import required layers and model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense

# Size of the word embedding vectors
embedding_dim = 64

# Create the Sequential model
model = Sequential([

    # Embedding layer converts word IDs into dense vectors
    # Note: input_length is removed to clear the deprecation warning
    Embedding(
        input_dim=vocab_size,      # Vocabulary size
        output_dim=embedding_dim   # Embedding vector size
    ),

    # Simple RNN layer with 64 neurons
    SimpleRNN(64),

    # Output layer with Softmax activation for 4 classes
    Dense(num_classes, activation='softmax')

])

# Explicitly build the model structure to show parameter counts
# None represents a variable batch size
model.build(input_shape=(None, max_length))

# Display the model architecture
model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 50, 64)         │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 4)              │           260 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,288,516 (4.92 MB)

 Trainable params: 1,288,516 (4.92 MB)

 Non-trainable params: 0 (0.00 B)

Step 7: Compile and Train the Model

In [9]:
# Configure the model for training
model.compile(

    # Optimizer adjusts the weights during training
    optimizer='adam',

    # Loss function for multi-class classification
    loss='categorical_crossentropy',

    # Metric used to evaluate the model
    metrics=['accuracy']
)

# Train the model
history = model.fit(

    # Training features
    X_train,

    # Training labels
    y_train,

    # Number of complete passes through the dataset
    epochs=5,

    # Number of samples processed before updating weights
    batch_size=64,

    # Use 20% of training data for validation
    validation_split=0.2
)

Epoch 1/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 44s 28ms/step - accuracy: 0.8304 - loss: 0.5054 - val_accuracy: 0.8670 - val_loss: 0.4096
Epoch 2/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 42s 28ms/step - accuracy: 0.9100 - loss: 0.2967 - val_accuracy: 0.8547 - val_loss: 0.4434
Epoch 3/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 81s 27ms/step - accuracy: 0.9266 - loss: 0.2436 - val_accuracy: 0.8537 - val_loss: 0.4797
Epoch 4/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 41s 27ms/step - accuracy: 0.9381 - loss: 0.1989 - val_accuracy: 0.8728 - val_loss: 0.3998
Epoch 5/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 42s 28ms/step - accuracy: 0.9509 - loss: 0.1576 - val_accuracy: 0.8697 - val_loss: 0.4994


Step 8: Evaluate the Model

In [10]:
# Evaluate the trained model on the test dataset
loss, accuracy = model.evaluate(
    X_test,
    y_test
)

# Print the test accuracy
print("Test Accuracy:", accuracy)

238/238 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.8733 - loss: 0.4948
Test Accuracy: 0.8732894659042358


----------------